In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("videogamesales.csv")
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df['Year'] = df['Year'].astype(int)
df['Top_Region'] = df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].idxmax(axis=1)

df.head()
top_publishers = df["Publisher"].value_counts().head(8).index
sankey_df = df[df["Publisher"].isin(top_publishers)].copy()

flow_data = sankey_df.groupby(["Publisher", "Genre"]).size().reset_index(name="count")

publishers = list(flow_data["Publisher"].unique())
genres = list(flow_data["Genre"].unique())
labels = publishers + genres
label_to_index = {label: i for i, label in enumerate(labels)}

source = flow_data["Publisher"].map(label_to_index).tolist()
target = flow_data["Genre"].map(label_to_index).tolist()
value = flow_data["count"].tolist()

color_list = [
    "#1C29E3", "#F02C0A", "#0BCD8F", "#6615BC",
    "#F3EF08", "#10A3BD", "#EF7111", "#25C04E"
]
publisher_colors = {
    publisher: color_list[i % len(color_list)]
    for i, publisher in enumerate(publishers)
}

genre_leader = (
    sankey_df.groupby(["Genre", "Publisher"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .drop_duplicates("Genre")
    .set_index("Genre")["Publisher"]
    .to_dict()
)

node_colors = (
    [publisher_colors[p] for p in publishers] +
    [publisher_colors[genre_leader[g]] for g in genres]
)

link_colors = [
    publisher_colors[flow_data.iloc[i]["Publisher"]]
    for i in range(len(flow_data))
]

fig = go.Figure(go.Sankey(
    node=dict(
        label=labels,
        pad=15,
        thickness=20,
        color=node_colors
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_colors
    )
))

fig.update_layout(title_text="Publisher vs Genre Sankey Diagram")
fig.show()
fig.write_html('sankey_export.html')